# N1 · checkpoint 策略: full vs sharded vs async

> 复用 `src/capstone_ckpt_recovery.py` · 大规模训练要定期存 checkpoint (容错)。
> 不同策略的开销 (阻塞/恢复/浪费) 怎么权衡? 看模拟 (接本专题讲义)。

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import numpy as np
import capstone_ckpt_recovery as ck
print('storage-dataops src 就绪')

## 1. 三种 checkpoint 策略的开销对比

In [ ]:
import matplotlib, matplotlib.pyplot as plt
matplotlib.rcParams['axes.unicode_minus']=False
for f in ['Microsoft YaHei','SimHei','DejaVu Sans']:
    try: matplotlib.rcParams['font.sans-serif']=[f]; break
    except Exception: pass
strategies = ['full','sharded','async']
rows = [ck.total_overhead(s) for s in strategies]
print(f"{'策略':<8} {'单次(s)':>9} {'阻塞(min)':>11} {'恢复(h)':>9} {'浪费%':>8}")
for r in rows:
    print(f"{r['strategy']:<8} {r['per_ckpt_s']:>9} {r['blocking_total_min']:>11} {r['recovery_total_h']:>9} {r['wasted_pct']:>7}%")
import numpy as np
labels=strategies; block=[r['blocking_total_min'] for r in rows]; waste=[r['wasted_pct'] for r in rows]
x=np.arange(len(labels)); w=0.35
fig,ax=plt.subplots(figsize=(7,4.2))
ax.bar(x-w/2, block, w, label='阻塞时间 (min)', color='C3')
ax2=ax.twinx(); ax2.bar(x+w/2, waste, w, label='浪费比例 %', color='C0')
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('阻塞 (min)', color='C3'); ax2.set_ylabel('浪费 %', color='C0')
ax.set_title('checkpoint 策略: 阻塞 vs 浪费 权衡'); plt.tight_layout(); plt.show()
print('→ full 简单但阻塞久; sharded 分片省阻塞; async 异步几乎不阻塞 (本专题: 按规模选策略)。')

> 本专题其余 src (`dataloader`/`sharding`/`webdataset_style`/`checkpoint`) 是数据管线与分片的模拟。